In [1]:
import dill as pickle
import matplotlib.pyplot as plt
import numpy as np
import os
import argparse
import matplotlib.gridspec as gridspec
import numpy as np
import subprocess
import numpy.random as rng
import pandas as pd
import random
import multiprocessing as mp
from scipy.stats import chi2
from scipy.linalg import eigh
import copy
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from scipy.spatial.distance import cdist    
from pathlib import Path
from collections import OrderedDict
from typing import Tuple, List, Union
from multiprocessing import Pool, cpu_count
from scipy.stats import qmc
from scipy.interpolate import interp1d
from numpy import linalg as LA

from matplotlib import cm
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from scipy.spatial.distance import cdist

from aero_optim.mf_sm.mf_infill import compute_pareto
from aero_optim.ffd.ffd import DLR_POD_2D

# plt.rcParams.update({
#     "text.usetex": True,
#     "font.family": "Times",
#     "figure.dpi": 200,
#     "font.size": 4,
#     'legend.fontsize': 4, 
#     "axes.titlesize": 4,
#     "axes.labelsize": 4
# })

# Configuration parameters
n_gen = 50
pop_size = 30

beta_ADP_bounds = [-0.67,0.93]  # degrees
beta_OP1_bounds = [-0.22,2.78]  # degrees
beta_OP2_bounds = [-2.67,0.33]  # degrees

hf_bsl_w_ADP = 0.03493
hf_bsl_w_OP2 = 0.05888
hf_bsl_w_OP1 = 0.03765
hf_bsl_w_OP  = (hf_bsl_w_OP1 + hf_bsl_w_OP2) * 0.5

# non-adapted fine mesh mixed-out los
lf_bsl_w_ADP = 0.07827
lf_bsl_w_OP1 = 0.21933
lf_bsl_w_OP2 = 0.07854
lf_bsl_w_OP  = (lf_bsl_w_OP1 + lf_bsl_w_OP2) * 0.5

baseline_file = "/home/mciarlatani/GPROfficial/beta-aero-optim/Optimization/RANS_bruteForce/ogv1c.dat"
baseline = np.loadtxt(baseline_file, skiprows=2) * 1e-3

OD = OrderedDict  # Type alias

In [2]:
def read_hf_results(home_dir, master_dir, n_DOE, n_infill):

    directoryList = [os.path.join(home_dir, 'hf_doe')] + [os.path.join(home_dir, master_dir+f'_{n}/high_infill_{n}') for n in range(n_infill)]

    df_temp = pd.DataFrame({
    "infill": pd.Series(dtype="int"),
    "cand": pd.Series(dtype="int"),
    "w_ADP": pd.Series(dtype="float"),
    "w_OP": pd.Series(dtype="float"),
    "w_OP1": pd.Series(dtype="float"),
    "w_OP2": pd.Series(dtype="float"),
    "beta_ADP": pd.Series(dtype="float"),
    "beta_OP1": pd.Series(dtype="float"),
    "beta_OP2": pd.Series(dtype="float"),
    "x1": pd.Series(dtype="float"),
    "x2": pd.Series(dtype="float"),
    "x3": pd.Series(dtype="float"),
    "x4": pd.Series(dtype="float"),
    "x5": pd.Series(dtype="float"),
    "fName": pd.Series(dtype="str")
    })
    cont = 0
    for dir in directoryList:
        cand = np.genfromtxt(os.path.join(dir, "candidates.txt"))
        if 'doe' in dir:
            for i in range(n_DOE):
                temp_ADP = np.genfromtxt(os.path.join(dir,f'MUSICAA/musicaa_g0_c{i}/ADP/QoI_convergence.csv'),skip_header=1,delimiter=',')[-1,:]
                temp_OP1 = np.genfromtxt(os.path.join(dir,f'MUSICAA/musicaa_g0_c{i}/OP1/QoI_convergence.csv'),skip_header=1,delimiter=',')[-1,:]
                temp_OP2 = np.genfromtxt(os.path.join(dir,f'MUSICAA/musicaa_g0_c{i}/OP2/QoI_convergence.csv'),skip_header=1,delimiter=',')[-1,:]
                df_temp.loc[len(df_temp)] = [0, i+1, temp_ADP[0], (temp_OP1[0]+temp_OP2[0])*0.5, temp_OP1[0], temp_OP2[0], temp_ADP[1], temp_OP1[1], temp_OP2[1],
                                             cand[i,0], cand[i,1], cand[i,2], cand[i,3], cand[i,4],
                                        os.path.join(dir, f"FFD/ogv1c_g0_c{i}.dat")]
        else:
            cont+=1
            temp_ADP = np.genfromtxt(os.path.join(dir,f'MUSICAA/musicaa_g0_c0/ADP/QoI_convergence.csv'),skip_header=1,delimiter=',')[-1,:]
            temp_OP1 = np.genfromtxt(os.path.join(dir,f'MUSICAA/musicaa_g0_c0/OP1/QoI_convergence.csv'),skip_header=1,delimiter=',')[-1,:]
            temp_OP2 = np.genfromtxt(os.path.join(dir,f'MUSICAA/musicaa_g0_c0/OP2/QoI_convergence.csv'),skip_header=1,delimiter=',')[-1,:]
            df_temp.loc[len(df_temp)] = [cont, 1, temp_ADP[0], (temp_OP1[0]+temp_OP2[0])*0.5, temp_OP1[0], temp_OP2[0], temp_ADP[1], temp_OP1[1], temp_OP2[1],
                                            cand[0], cand[1], cand[2], cand[3], cand[4],
                                    os.path.join(dir, f"FFD/ogv1c_g0_c0.dat")]

    return df_temp

In [ ]:
r

In [ ]:
nLFtoHF = 10

n_DOE = 5
n_infill = 15
master_dir_mf = 'output_paper'
master_dir_hf = 'output_hf'

selected = [6, 0, 3]  # indices of chosen optimal candidates
plt.close('all')
# Load mixed-out loss results
mf_dir = "/home/mciarlatani/GPROfficial/beta-aero-optim/Optimization/irene_mf_DLR/" + master_dir_mf
hf_dir = "/home/mciarlatani/GPROfficial/beta-aero-optim/Optimization/irene_mf_DLR/" + master_dir_hf
lf_dir = "/home/mciarlatani/GPROfficial/beta-aero-optim/Optimization/RANS_bruteForce/output_24TE/"

mf_hf_opt = read_hf_results(mf_dir, master_dir_mf, n_DOE, n_infill)
hf_opt = read_hf_results(hf_dir, master_dir_hf, n_DOE, n_infill)

read_model_results: